
### Keept the database update as Kleio files are translated
Updates source translations and imports into database sources with no errors.

Only changed files since last import, or new files, will be processed.

In [ ]:
from timelink.notebooks import TimelinkNotebook

tlnb = TimelinkNotebook()
tlnb.print_info()


In [ ]:
import logging
import os
import time
logging.basicConfig(level=logging.INFO)
# show logging in the notebook
from IPython.display import display, HTML
display(HTML("<style>.jp-OutputArea-output { font-size: 1.5em; }</style>"))
# Set up logging to display in the notebook
logger = logging.getLogger()
logger.setLevel(logging.INFO)
handler = logging.StreamHandler()
handler.setLevel(logging.INFO)
formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
handler.setFormatter(formatter)
logger.addHandler(handler)
# Set up the logging configuration
# Set the logging level to INFO
# Set the format for the log messages
kleio_home_path = tlnb.kleio_server.get_kleio_home()
print(f"kleio_home_path: {kleio_home_path}")
# Monitor the directory for changes
def monitor_kleio_home(path):
    logging.info(f"Monitoring changes in {path}")
    # Initial scan of all files in subdirectories
    previous_files = {}
    for dirpath, _, filenames in os.walk(path):
        for filename in filenames:
            # Only monitor Kleio files
            if not filename.lower().endswith((".cli", ".kleio")):
                continue
            fullpath = os.path.join(dirpath, filename)
            try:
                previous_files[fullpath] = os.path.getmtime(fullpath)
            except OSError:
                continue
    while True:
        time.sleep(10)  # Check every 10 seconds
        # Scan again recursively
        current_files = {}
        for dirpath, _, filenames in os.walk(path):
            for filename in filenames:
                # Only monitor Kleio files
                if not filename.lower().endswith((".cli", ".kleio")):
                    continue
                fullpath = os.path.join(dirpath, filename)
                try:
                    current_files[fullpath] = os.path.getmtime(fullpath)
                except OSError:
                    continue
        changed_files = [f for f in current_files if f not in previous_files or current_files[f] != previous_files[f]]
        if changed_files:
            logging.info(f"Detected changes in files: {changed_files}")
            tlnb.update_from_sources(path='')
        previous_files = current_files

tlnb.db.update_from_sources(path='', with_import_errors=True)
monitor_kleio_home(kleio_home_path)